# WM-811K Term Project — Step 2

## 카테고리 1: 클래스 불균형 처리 Ablation (Student 모델 기준)

**설계 배경**: Teacher(ResNet-18)는 불균형 처리 없이도 macro-F1 0.89로 충분히 높아
불균형 처리의 효과가 잘 드러나지 않습니다. 따라서 경량 **Student CNN(0.39M)** 을 분석 대상으로
삼습니다. "큰 모델은 불균형 없이도 잘하지만, 경량 모델은 불균형 처리가 필요하다"는 것이
이 실험의 핵심 가설이며, 이는 KD(카테고리 2)와도 자연스럽게 연결됩니다.

**비교 대상** (모두 augment=False, 30 epoch, 동일 seed):
- `none`: 불균형 처리 없음 (baseline)
- `oversample`: WeightedRandomSampler (데이터 수준)
- `focal`: Focal Loss (알고리즘 수준)
- `classweight`: Class Weight (알고리즘 수준)

**평가**: Accuracy + macro-F1 + 클래스별 F1 (소수 클래스 변화가 핵심)

## 1. 환경 설정 (Step 1과 동일)

In [1]:
!nvidia-smi

Sun May 31 08:00:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/wm811k_term'
DATA_PATH = os.path.join(PROJECT_DIR, 'data', 'wm811k_64.npz')
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
assert os.path.exists(DATA_PATH), f'데이터가 없습니다: {DATA_PATH}'
print('데이터 확인됨:', DATA_PATH)

Mounted at /content/drive
데이터 확인됨: /content/drive/MyDrive/wm811k_term/data/wm811k_64.npz


In [3]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as tvm
from sklearn.metrics import f1_score
import json

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

set_seed(42)
device = get_device()
print('Device:', device, '| PyTorch:', torch.__version__)

Device: cuda | PyTorch: 2.11.0+cu128


In [4]:
d = np.load(DATA_PATH, allow_pickle=True)
X_train, y_train = d['X_train'], d['y_train']
X_test, y_test = d['X_test'], d['y_test']
CLASSES = list(d['classes'])
NUM_CLASSES = len(CLASSES)
print(f'Train: {X_train.shape}  Test: {X_test.shape}  Classes: {NUM_CLASSES}')

Train: (138360, 64, 64)  Test: (34590, 64, 64)  Classes: 9


## 2. 데이터 유틸 / 모델 / 학습 함수 (Step 1과 동일)

In [5]:
class WaferDataset(Dataset):
    def __init__(self, X, y, train=False, augment=True):
        self.X, self.y, self.train, self.augment = X, y, train, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img = self.X[i].astype(np.float32) / 2.0
        if self.train and self.augment:
            if random.random() < 0.5: img = np.fliplr(img).copy()
            if random.random() < 0.5: img = np.flipud(img).copy()
        return torch.from_numpy(img).unsqueeze(0), int(self.y[i])

def compute_class_weights(y, num_classes=NUM_CLASSES):
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = 1.0 / counts
    return torch.tensor(w / w.sum() * num_classes, dtype=torch.float32)

def make_loaders(X_tr, y_tr, X_te, y_te, batch_size=128, oversample=False, augment=False):
    tr_ds = WaferDataset(X_tr, y_tr, train=True, augment=augment)
    te_ds = WaferDataset(X_te, y_te, train=False, augment=False)
    if oversample:
        counts = np.bincount(y_tr, minlength=NUM_CLASSES).astype(np.float64)
        counts[counts == 0] = 1.0
        sample_w = (1.0 / counts)[y_tr]
        sampler = WeightedRandomSampler(torch.tensor(sample_w, dtype=torch.double), len(y_tr), replacement=True)
        tr = DataLoader(tr_ds, batch_size=batch_size, sampler=sampler, num_workers=8, pin_memory=True)
    else:
        tr = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True)
    te = DataLoader(te_ds, batch_size=256, shuffle=False, num_workers=8, pin_memory=True)
    return tr, te

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__(); self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1); p = torch.exp(logp)
        logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = -((1 - p_t) ** self.gamma) * logp_t
        if self.alpha is not None:
            loss = self.alpha.to(logits.device)[target] * loss
        return loss.mean()

class StudentCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(1,32), block(32,64), block(64,128), block(128,256))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.features(x); x = self.gap(x).flatten(1); return self.classifier(x)

def get_student(num_classes=NUM_CLASSES):
    return StudentCNN(num_classes)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def train_one_epoch(model, loader, optimizer, criterion, use_amp=True):
    model.train(); total=correct=0; loss_sum=0.0
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            out = model(x); loss = criterion(out, y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        loss_sum += loss.item()*x.size(0); correct += (out.argmax(1)==y).sum().item(); total += x.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds=[]; trues=[]
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        preds.append(model(x).argmax(1).cpu().numpy()); trues.append(y.numpy())
    pred, true = np.concatenate(preds), np.concatenate(trues)
    acc = (pred==true).mean()
    macro_f1 = f1_score(true, pred, average='macro', zero_division=0)
    per_class = f1_score(true, pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    return {'accuracy':float(acc),'macro_f1':float(macro_f1),'per_class_f1':per_class.tolist()}

def print_per_class(per_class_f1):
    print('  Per-class F1:')
    for c, f in zip(CLASSES, per_class_f1): print(f'    {c:<12} {f:.4f}')

print('Student params:', f'{count_params(get_student()):,}')
print('유틸/모델/학습 함수 정의 완료')

Student params: 390,633
유틸/모델/학습 함수 정의 완료


## 3. Student 학습 함수 (불균형 처리 옵션)
Step 1의 `run_teacher`와 동일 구조이나 **Student 모델** 사용.
Student는 작아서 lr=0.05, epochs=30으로 설정.

In [6]:
def run_student(imbalance='none', augment=False, epochs=30, batch=128, lr=0.05, tag=None):
    set_seed(42)
    if tag is None:
        tag = f"student_{imbalance}"
    oversample = (imbalance == 'oversample')
    tr_loader, te_loader = make_loaders(X_train, y_train, X_test, y_test, batch, oversample, augment)

    if imbalance == 'focal':
        criterion = FocalLoss(gamma=2.0)
    elif imbalance == 'classweight':
        criterion = nn.CrossEntropyLoss(weight=compute_class_weights(y_train).to(device))
    else:
        criterion = nn.CrossEntropyLoss()

    model = get_student().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ckpt_path = os.path.join(CKPT_DIR, f'{tag}.pt')

    print(f'>>> 학습 시작: {tag} (imbalance={imbalance}, augment={augment}, epochs={epochs})\n')
    history, best_f1 = [], 0.0
    for ep in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, tr_loader, optimizer, criterion)
        scheduler.step()
        m = evaluate(model, te_loader)
        history.append({'epoch':ep,'train_loss':tr_loss,'test_acc':m['accuracy'],
                        'macro_f1':m['macro_f1'],'per_class_f1':m['per_class_f1']})
        print(f"Epoch {ep:02d} | loss {tr_loss:.3f} | test_acc {m['accuracy']:.4f} | macro_f1 {m['macro_f1']:.4f}")
        if m['macro_f1'] > best_f1:
            best_f1 = m['macro_f1']; torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    final = evaluate(model, te_loader)
    print(f"\n{'='*50}\n[{tag}] 최종 (best macro-F1 모델)")
    print(f"  Accuracy : {final['accuracy']*100:.2f}%\n  Macro-F1 : {final['macro_f1']:.4f}")
    print_per_class(final['per_class_f1'])
    with open(os.path.join(RESULTS_DIR, f'{tag}.json'), 'w') as f:
        json.dump({'imbalance':imbalance,'augment':augment,'accuracy':final['accuracy'],
                   'macro_f1':final['macro_f1'],'per_class_f1':final['per_class_f1'],
                   'classes':CLASSES,'history':history}, f, indent=2)
    print(f"\n저장 완료 → {tag}.pt / {tag}.json")
    return final

print('run_student 정의 완료')

run_student 정의 완료


## 4. 불균형 처리 Ablation 실행
4개를 순서대로 실행합니다. Student는 가벼워서 각 30 epoch이 빠릅니다.
**한 셀에서 전부 실행** — 중간에 끊겨도 각 결과가 Drive에 저장됩니다.

In [7]:
results = {}
for imb in ['none', 'oversample', 'focal', 'classweight']:
    print(f"\n{'#'*60}\n# {imb}\n{'#'*60}")
    results[imb] = run_student(imbalance=imb, augment=False, epochs=30)


############################################################
# none
############################################################
>>> 학습 시작: student_none (imbalance=none, augment=False, epochs=30)

Epoch 01 | loss 0.237 | test_acc 0.9469 | macro_f1 0.6392
Epoch 02 | loss 0.150 | test_acc 0.5936 | macro_f1 0.4249
Epoch 03 | loss 0.132 | test_acc 0.9572 | macro_f1 0.6866
Epoch 04 | loss 0.122 | test_acc 0.9582 | macro_f1 0.6963
Epoch 05 | loss 0.115 | test_acc 0.9010 | macro_f1 0.6831
Epoch 06 | loss 0.110 | test_acc 0.9643 | macro_f1 0.7733
Epoch 07 | loss 0.105 | test_acc 0.9620 | macro_f1 0.7745
Epoch 08 | loss 0.100 | test_acc 0.9570 | macro_f1 0.7420
Epoch 09 | loss 0.096 | test_acc 0.9501 | macro_f1 0.7067
Epoch 10 | loss 0.092 | test_acc 0.9674 | macro_f1 0.8214
Epoch 11 | loss 0.089 | test_acc 0.9515 | macro_f1 0.7349
Epoch 12 | loss 0.084 | test_acc 0.9466 | macro_f1 0.7383
Epoch 13 | loss 0.082 | test_acc 0.9621 | macro_f1 0.7422
Epoch 14 | loss 0.077 | test_acc 0.9373 | macro_

## 5. 결과 요약 비교

In [8]:
print(f"{'기법':<14}{'Accuracy':<12}{'Macro-F1':<12}")
print('-'*38)
for imb in ['none', 'oversample', 'focal', 'classweight']:
    r = results[imb]
    print(f"{imb:<14}{r['accuracy']*100:>8.2f}%   {r['macro_f1']:>8.4f}")

print(f"\n{'클래스별 F1 비교':}")
print(f"{'클래스':<12}" + ''.join(f'{imb[:8]:<10}' for imb in ['none','oversample','focal','classweight']))
print('-'*52)
for ci, c in enumerate(CLASSES):
    row = f'{c:<12}'
    for imb in ['none','oversample','focal','classweight']:
        row += f"{results[imb]['per_class_f1'][ci]:<10.3f}"
    print(row)

기법            Accuracy    Macro-F1    
--------------------------------------
none             97.27%     0.8606
oversample       96.83%     0.8360
focal            97.21%     0.8569
classweight      96.30%     0.8527

클래스별 F1 비교
클래스         none      oversamp  focal     classwei  
----------------------------------------------------
Center      0.914     0.895     0.913     0.901     
Donut       0.823     0.781     0.792     0.815     
Edge-Loc    0.795     0.781     0.792     0.767     
Edge-Ring   0.974     0.969     0.973     0.971     
Loc         0.716     0.636     0.712     0.690     
Near-full   0.933     0.921     0.951     0.968     
Random      0.879     0.862     0.866     0.877     
Scratch     0.724     0.691     0.725     0.703     
none        0.989     0.987     0.988     0.983     


---
### Step 2 완료 후
- **none(baseline) 대비 oversample/focal/classweight의 macro-F1 향상**을 확인하세요.
- 특히 **소수 클래스(Near-full, Donut, Scratch, Loc)의 F1 변화**가 핵심입니다.
- 데이터 수준(oversample) vs 알고리즘 수준(focal/classweight) 중 어느 쪽이 효과적인지 분석합니다.
- 가장 좋은 기법을 다음 단계(KD)의 Teacher 학습에 적용합니다.